In [2]:
import verl
print(verl.__version__)

0.7.0


In [3]:
import os

# 获取当前工作目录
current_path = os.getcwd()
print(current_path)

c:\Users\刘心怡\Desktop\大模型春招\llms\project\qwen-math-rl\data


In [5]:
cd ..

c:\Users\刘心怡\Desktop\大模型春招\llms\project\qwen-math-rl


c:\Users\刘心怡\Desktop\大模型春招\llms\project\qwen-math-rl\.venv\lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [6]:
cd verl/examples/data_preprocess

c:\Users\刘心怡\Desktop\大模型春招\llms\project\qwen-math-rl\verl\examples\data_preprocess


c:\Users\刘心怡\Desktop\大模型春招\llms\project\qwen-math-rl\.venv\lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [ ]:
import os

output_dir = "../outputs/gsm8k_processed"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

In [5]:
from datasets import load_dataset
from datasets import load_from_disk

# 加载训练集
dataset = load_from_disk('./gsm8k_dataset/train')

In [6]:
# 查看数据集字段名
print(dataset[0])  # 打印第一个样本，检查字段结构

{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}


In [ ]:
import argparse

# 解析命令行参数
parser = argparse.ArgumentParser()
parser.add_argument('--data_dir', type=str, required=True, help="Path to the dataset")
args = parser.parse_args()


In [ ]:
from gsm8k import compute_score  # 如果 gsm8k.py 中的函数已正确封装

# 假设 dataset 是加载的数据集
for example in dataset:
    solution_str = example['reasoning']  # 或者使用对应的字段
    ground_truth = example['answer']  # 根据数据集结构选择正确字段
    score = compute_score(solution_str, ground_truth, method="strict")
    print(f"Score: {score}")

In [10]:
!python -c "import sys; print(sys.executable)"
!python -m pip show datasets
!python -m pip show pyarrow

c:\Users\刘心怡\Desktop\大模型春招\llms\project\qwen-math-rl\.venv\Scripts\python.exe
Name: datasets
Version: 4.5.0
Summary: HuggingFace community-driven open-source library of datasets
Home-page: https://github.com/huggingface/datasets
Author: HuggingFace Inc.
Author-email: thomas@huggingface.co
License: Apache 2.0
Location: c:\users\刘心怡\desktop\大模型春招\llms\project\qwen-math-rl\.venv\lib\site-packages
Requires: dill, filelock, fsspec, httpx, huggingface-hub, multiprocess, numpy, packaging, pandas, pyarrow, pyyaml, requests, tqdm, xxhash
Required-by: verl
Name: pyarrow
Version: 23.0.1
Summary: Python library for Apache Arrow
Home-page: https://arrow.apache.org/
Author: 
Author-email: 
License-Expression: Apache-2.0
Location: c:\users\刘心怡\desktop\大模型春招\llms\project\qwen-math-rl\.venv\lib\site-packages
Requires: 
Required-by: datasets, verl


In [9]:
!python gsm8k.py -h

usage: gsm8k.py [-h] [--local_dir LOCAL_DIR] [--hdfs_dir HDFS_DIR]
                [--local_dataset_path LOCAL_DATASET_PATH]
                [--local_save_dir LOCAL_SAVE_DIR]

options:
  -h, --help            show this help message and exit
  --local_dir LOCAL_DIR
                        The save directory for the preprocessed dataset.
  --hdfs_dir HDFS_DIR
  --local_dataset_path LOCAL_DATASET_PATH
                        The local path to the raw dataset, if it exists.
  --local_save_dir LOCAL_SAVE_DIR
                        The save directory for the preprocessed dataset.


In [12]:
!python gsm8k.py --local_dataset_path openai/gsm8k --local_save_dir ./output

c:\Users\刘心怡\Desktop\大模型春招\llms\project\qwen-math-rl\.venv\lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\hf_cache\hub\datasets--openai--gsm8k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)

Generating train split: 100%|██████████| 7473/7473 [00:00<00:00, 115459.55 examples/s]

Generating test split: 100%|

In [16]:
import pandas as pd

df = pd.read_parquet("./output/train.parquet")

print("列名：", df.columns.tolist())
print("shape:", df.shape)
print(df.dtypes)

列名： ['data_source', 'prompt', 'ability', 'reward_model', 'extra_info']
shape: (7473, 5)
data_source     object
prompt          object
ability         object
reward_model    object
extra_info      object
dtype: object


In [18]:
sample = df.iloc[0]
print(sample)
print(f"总样本数: {len(df)}")

data_source                                          openai/gsm8k
prompt          [{'content': 'Natalia sold clips to 48 of her ...
ability                                                      math
reward_model              {'ground_truth': '72', 'style': 'rule'}
extra_info      {'answer': 'Natalia sold 48/2 = <<48/2=24>>24 ...
Name: 0, dtype: object
总样本数: 7473
